In [1]:
import pandas as pd

df = pd.read_csv('D:/Downloads/APT_combined.csv', low_memory=False)

In [2]:
df.value_counts('Activity')

Activity
Maintain Access                              27128
Encrypted Channel: Symmetric Cryptography    27111
Data Transfer Size Limits                     6988
Remote System Discovery                       1002
Exfiltration over C2 channel                   446
Remove Traces                                  362
Unsecured Credentials                           88
Name: count, dtype: int64

In [3]:
# Ánh xạ hành động với chỉ số ID
action_mapping = {
    'Maintain Access': 0,
    'Encrypted Channel: Symmetric Cryptography': 1,     
    'Data Transfer Size Limits': 2,
    'Remote System Discovery': 3,
    'Exfiltration over C2 channel': 4,
    'Remove Traces': 5,
    'Unsecured Credentials': 6 
}

In [4]:
# Hàm tính điểm dựa trên network flow
def score_network_flow(row):
    action_scores = []

    # Kiểm tra Maintain Access (duy trì kết nối lâu)
    if row.get("bidirectional_duration_ms", 0) > 300000:
        action_scores.append(action_mapping['Maintain Access'])

    # Kiểm tra mã hóa TLS
    if row.get("protocol", "") in ["TLS", "SSL"]:
        action_scores.append(action_mapping['Encrypted Channel: Symmetric Cryptography'])

    # Kiểm tra giới hạn dữ liệu tải lên
    if row.get("bidirectional_bytes", 0) < 10 * 1024 * 1024:
        action_scores.append(action_mapping['Data Transfer Size Limits'])

    # Kiểm tra hoạt động dò quét hệ thống
    if any(cmd in str(row.get("application_name", "")).lower() for cmd in ["whoami", "ipconfig", "netstat"]):
        action_scores.append(action_mapping['Remote System Discovery'])

    # Kiểm tra Exfiltration over C2 (rò rỉ dữ liệu đến server)
    if row.get("dst_port", 0) in [443, 8080, 53] and row.get("bidirectional_bytes", 0) > 1000000:
        action_scores.append(action_mapping['Exfiltration over C2 channel'])

    # Kiểm tra Remove Traces (xóa log, che dấu dấu vết)
    if any(cmd in str(row.get("application_name", "")).lower() for cmd in ["rm", "delete", "wipe"]):
        action_scores.append(action_mapping['Remove Traces'])

    # Kiểm tra Unsecured Credentials (gửi mật khẩu dạng plaintext)
    if any(keyword in str(row.get("user_agent", "")).lower() for keyword in ["password=", "username="]):
        action_scores.append(action_mapping['Unsecured Credentials'])

    return list(set(action_scores))  # Loại bỏ trùng lặp

In [5]:
# Áp dụng scoring cho toàn bộ dataset
df['Score_Actions'] = df.apply(score_network_flow, axis=1)


In [6]:
# Xuất kết quả ra file CSV
df.to_csv("D:/Downloads/APT_Scored.csv", index=False)

print("✅ Scoring hoàn tất! Kết quả đã được lưu vào APT_Scored.csv")

✅ Scoring hoàn tất! Kết quả đã được lưu vào APT_Scored.csv
